## Visualization and stats

In [14]:
import numpy as np 

features = np.load("../features/bipolaire/rem_only/AE129/AE129_REM_1_features.npz")
features.files



['Delta_mean',
 'Delta_var',
 'Delta_std',
 'Delta_skew',
 'Delta_power',
 'Delta_psd_mean',
 'Delta_wavelet',
 'Delta_pca_var',
 'Delta_ica_infomax_energy',
 'Theta_mean',
 'Theta_var',
 'Theta_std',
 'Theta_skew',
 'Theta_power',
 'Theta_psd_mean',
 'Theta_wavelet',
 'Theta_pca_var',
 'Theta_ica_infomax_energy',
 'Alpha_mean',
 'Alpha_var',
 'Alpha_std',
 'Alpha_skew',
 'Alpha_power',
 'Alpha_psd_mean',
 'Alpha_wavelet',
 'Alpha_pca_var',
 'Alpha_ica_infomax_energy',
 'Beta_mean',
 'Beta_var',
 'Beta_std',
 'Beta_skew',
 'Beta_power',
 'Beta_psd_mean',
 'Beta_wavelet',
 'Beta_pca_var',
 'Beta_ica_infomax_energy',
 'Gamma_mean',
 'Gamma_var',
 'Gamma_std',
 'Gamma_skew',
 'Gamma_power',
 'Gamma_psd_mean',
 'Gamma_wavelet',
 'Gamma_pca_var',
 'Gamma_ica_infomax_energy']

In [15]:
len(features)

45

### Puissance spectrale


In [1]:
%matplotlib inline

import os, sys
from pathlib import Path
import numpy as np
import mne
import matplotlib.pyplot as plt

mne.set_log_level("INFO")   # "DEBUG" si tu veux plus verbeux

print("mne:", mne.__version__)


mne: 1.10.0


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Batch REM extraction (FIF + hypnogram .txt) + bipolar montage + per-channel Morlet TFR with autoscale.

- Cherche automatiquement les patients dans fif_root: {base}/*.fif ou directement *.fif à la racine
  (ou utilise une liste PATIENTS définie à la main).
- Charge le .fif sans preload, lit les annotations .txt via get_rem_annotations(base, annot_root).
- Extrait/concatène les segments REM, applique un **montage bipolaire** (MYMONTAGE_BIP), met l'EEG en µV.
- Calcule une TFR (Morlet) par canal EEG bipolaire, autoscale robuste (percentiles), évite les figures blanches.
- Sauvegarde:
    - out_root/{base}/{base}_REM_concat_uV.fif (optionnel)
    - out_root/{base}/{base}_{canal}_tfr_REM.png

Montage bipolaire souhaité (MYMONTAGE_BIP):
  EOGD-A1 = EOGD - A1
  EOGG-A1 = EOGG - A1
  Fp2-C4  = Fp2  - C4
  C4-O2   = C4   - O2
  T4-O2   = T4   - O2
  Cz-Pz   = Cz   - Pz
  Fp1-C3  = Fp1  - C3
  C3-O1   = C3   - O1
  Fp1-T3  = Fp1  - T3
  T3-O1   = T3   - O1
Conservation (non bipolaire): Menton, JAMBG, JAMBD, RONF, EMG1, EMG2, ECG
Canaux de type EEG = {Fp2-C4, C4-O2, T4-O2, Cz-Pz, Fp1-C3, C3-O1, Fp1-T3, T3-O1}
"""

import os
from pathlib import Path
import numpy as np
import mne

# Matplotlib non-interactif si lancé en script
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:
    import matplotlib.pyplot as plt

# ===================== PARAMÈTRES GLOBAUX =====================
fif_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/full")  # contient {base}/*.fif ou *.fif
annot_root  = Path("/Volumes/Crucial X6/EEG/raw")                          # {base}/ avec hypnogrammes .txt
out_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")

# Laisser à None pour auto-découverte, ou donner une liste:
# - soit ["AE129","BJ138",...]
# - soit [("AE129", "/chemin/vers/fichier.fif"), ...] pour pointer un .fif précis
PATIENTS    = None

# TFR
freq_min    = 1.0
freq_max    = 40.0
n_freqs     = 30
cycles_mult = 0.5
epoch_dur   = 4.0
decim       = 2
cmap        = "jet"
vmin, vmax  = None, None       # None => autoscale via percentiles ci-dessous
auto_pct    = (5, 95)

# I/O options
save_rem_fif   = True
overwrite_figs = False

# Après re-référencement bipolaire, on ne filtre PAS par nom "EEG"
RESTRICT_TO_NAME_WITH_EEG = False
# ===================== /PARAMS =====================


# ---- Import de la fonction d'annotations (fallback si besoin) ----
def _ensure_get_rem_annotations():
    try:
        from src.annotations import get_rem_annotations  # version officielle
        return get_rem_annotations
    except Exception:
        import pandas as pd
        from pathlib import Path
        def load_annotation_file(txt_path):
            df = pd.read_csv(txt_path, sep="\t", names=["start", "temps", "stage", "index"])
            df = df.dropna(subset=["start", "stage"])
            df["duration"] = df["start"].shift(-1) - df["start"]
            df = df[:-1]
            rem_df = df[df["stage"].str.upper().str.strip() == "REM"]
            return rem_df[["start", "duration"]].values

        def get_rem_annotations(base_name, annot_dir):
            patient_code = base_name.split("_")[0]
            txt_dir = Path(annot_dir) / patient_code
            txt_candidates = list(txt_dir.glob("*.txt"))
            for txt_path in txt_candidates:
                try:
                    rem_intervals = load_annotation_file(txt_path)
                    if len(rem_intervals) > 0:
                        return mne.Annotations(
                            onset=[float(s) for s, _ in rem_intervals],
                            duration=[float(d) for _, d in rem_intervals],
                            description=["REM"] * len(rem_intervals)
                        )
                except Exception:
                    continue
            return None
        return get_rem_annotations

get_rem_annotations = _ensure_get_rem_annotations()


def _sanitize(name: str) -> str:
    return "".join(c for c in name if c.isalnum() or c in ("_", "-")).replace(" ", "")


def discover_patients(fif_root: Path):
    """Retourne une liste [(base, fif_path), ...] en cherchant des .fif.
    - Priorité: sous-dossiers {base}/*.fif (prend le .fif le plus gros)
    - Fallback: fichiers *.fif directement dans fif_root (base = préfixe avant le premier "_")
    """
    mapping = {}
    # 1) sous-dossiers {base}/*.fif
    for child in sorted(fif_root.iterdir()):
        if not child.is_dir():
            continue
        base = child.name
        fif_candidates = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if not fif_candidates:
            continue
        fif_path = max(fif_candidates, key=lambda p: p.stat().st_size)
        mapping[base] = fif_path

    # 2) fichiers *.fif à la racine
    root_fifs = [p for p in fif_root.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
    for p in root_fifs:
        base = p.stem.split("_")[0]
        cur = mapping.get(base)
        if cur is None or p.stat().st_size > cur.stat().st_size:
            mapping[base] = p

    items = sorted(mapping.items())  # [(base, path), ...]
    return items


# ===================== BIPOLAIRE =====================

def _safe_bipolar(inst: mne.io.BaseRaw, anode: str, cathode: str, new_name: str, base: str) -> bool:
    """Crée un canal bipolaire anode-cathode ssi les deux existent."""
    if anode in inst.ch_names and cathode in inst.ch_names:
        try:
            mne.set_bipolar_reference(
                inst, anode=anode, cathode=cathode, ch_name=new_name,
                drop_refs=False, copy=False, verbose="ERROR"
            )
            return True
        except Exception as e:
            print(f"[{base}] Bipolaire {new_name} échec: {e}")
    else:
        missing = [x for x in (anode, cathode) if x not in inst.ch_names]
        print(f"[{base}] Bipolaire {new_name} ignoré (manque {missing})")
    return False


def _apply_bipolar_montage(inst: mne.io.BaseRaw, base: str) -> None:
    """Applique le montage bipolaire MYMONTAGE_BIP + typage des canaux + réduction au set voulu."""
    pairs = [
        ("EOGD", "A1",  "EOGD-A1"),
        ("EOGG", "A1",  "EOGG-A1"),
        ("Fp2",  "C4",  "Fp2-C4"),
        ("C4",   "O2",  "C4-O2"),
        ("T4",   "O2",  "T4-O2"),
        ("Cz",   "Pz",  "Cz-Pz"),
        ("Fp1",  "C3",  "Fp1-C3"),
        ("C3",   "O1",  "C3-O1"),
        ("Fp1",  "T3",  "Fp1-T3"),
        ("T3",   "O1",  "T3-O1"),
    ]

    created = []
    for a, c, n in pairs:
        if _safe_bipolar(inst, a, c, n, base):
            created.append(n)

    # Typage: EEG pour paires EEG, EOG pour EOG*, EMG/ECG pour les capteurs conservés
    eeg_bip = ["Fp2-C4", "C4-O2", "T4-O2", "Cz-Pz", "Fp1-C3", "C3-O1", "Fp1-T3", "T3-O1"]
    eog_bip = ["EOGD-A1", "EOGG-A1"]
    keep_raw = ["Menton", "JAMBG", "JAMBD", "RONF", "EMG1", "EMG2", "ECG"]

    type_map = {}
    for ch in eeg_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eeg"
    for ch in eog_bip:
        if ch in inst.ch_names:
            type_map[ch] = "eog"
    for ch in ("Menton", "EMG1", "EMG2", "JAMBG", "JAMBD"):
        if ch in inst.ch_names:
            type_map[ch] = "emg"
    if "ECG" in inst.ch_names:
        type_map["ECG"] = "ecg"
    if "RONF" in inst.ch_names:
        type_map["RONF"] = "misc"

    if type_map:
        try:
            inst.set_channel_types(type_map)
        except Exception as e:
            print(f"[{base}] set_channel_types après bipolaire: {e}")

    # Réduire strictement aux canaux voulus (ceux créés + capteurs conservés)
    desired = eog_bip + eeg_bip + keep_raw
    present = [ch for ch in desired if ch in inst.ch_names]
    if not present:
        print(f"[{base}] Aucun canal bipolaire/utile présent après montage → rien à faire")
        return
    inst.pick(present)

    print(f"[{base}] Montage bipolaire créé. Canaux conservés ({len(inst.ch_names)}): {inst.ch_names}")


# ===================== /BIPOLAIRE =====================


def _find_fif_for_base(base: str) -> Path | None:
    """Cherche un .fif pour `base` dans fif_root/{base}/*.fif, sinon à la racine par préfixe du nom."""
    # 1) dossier du patient
    child = fif_root / base
    if child.is_dir():
        cand = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if cand:
            return max(cand, key=lambda p: p.stat().st_size)
    # 2) racine, par préfixe stem
    cand = [p for p in fif_root.glob(f"{base}*.fif") if p.is_file() and not p.name.startswith("._")]
    if cand:
        return max(cand, key=lambda p: p.stat().st_size)
    return None


def process_one_patient(item):
    # item peut être "base" (str) ou (base, fif_path)
    if isinstance(item, tuple):
        base, fif_path = item
        fif_path = Path(fif_path)
    else:
        base = str(item)
        fif_path = _find_fif_for_base(base)

    if fif_path is None or not Path(fif_path).exists():
        print(f"[{base}] FIF introuvable -> skip")
        return

    print(f"\n=== {base} ===")
    try:
        raw_full = mne.io.read_raw_fif(fif_path, preload=True, verbose="ERROR")
    except Exception as e:
        print(f"[{base}] Erreur lecture FIF: {e} -> skip")
        return
    print(raw_full)

    # Annotations REM depuis .txt
    rem_annots = get_rem_annotations(base, annot_dir=str(annot_root))
    if rem_annots is None or len(rem_annots) == 0:
        print(f"[{base}] Aucune annotation REM -> skip")
        return

    sfreq = float(raw_full.info["sfreq"])
    t_end = raw_full.times[-1]
    eps   = 1.0 / sfreq

    # Filtrage des segments valides
    valid = []
    for onset, dur, desc in zip(rem_annots.onset, rem_annots.duration, rem_annots.description):
        if str(desc).upper() != "REM":
            continue
        if dur is None or dur <= 0:
            continue
        tmin = max(0.0, float(onset))
        tmax = min(tmin + float(dur), t_end) - eps
        if tmax <= tmin:
            continue
        valid.append((tmin, tmax))

    print(f"[{base}] Segments REM valides: {len(valid)}")
    if not valid:
        print(f"[{base}] Aucun segment REM valide -> skip")
        return

    # Concaténation REM (on garde tout puis on applique le bipolaire)
    rem_raws = []
    for (tmin, tmax) in valid:
        try:
            seg = raw_full.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
            rem_raws.append(seg)
        except Exception as e:
            print(f"[{base}] Crop {tmin:.2f}-{tmax:.2f}s échoué: {e}")

    if not rem_raws:
        print(f"[{base}] Aucun segment ajouté -> skip")
        return

    rem_raw = mne.concatenate_raws(rem_raws, verbose="ERROR")
    print(rem_raw)

    # ===== Montage bipolaire puis typage =====
    _apply_bipolar_montage(rem_raw, base)

    # --- Ne garder QUE l'EEG par TYPE (les 8 bipolaires listés) ---
    try:
        rem_raw.pick_types(
            meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
            misc=False, resp=False, seeg=False, ecog=False, fnirs=False
        )
        if RESTRICT_TO_NAME_WITH_EEG:
            eeg_names = [ch for ch in rem_raw.ch_names if "EEG" in ch.upper()]
            if len(eeg_names) == 0:
                print(f"[{base}] Aucun canal avec 'EEG' dans le nom; conservez tous les EEG typés.")
            else:
                rem_raw.pick(eeg_names)
        print(f"[{base}] Canaux EEG retenus ({len(rem_raw.ch_names)}): {rem_raw.ch_names}")
    except Exception as e:
        print(f"[{base}] Échec du filtrage EEG-only: {e} -> skip")
        return

    # Scaling µV (toujours)
    try:
        rem_raw.load_data()
        eeg_picks = mne.pick_types(rem_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
        rem_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)
        if hasattr(rem_raw, "set_unit"):
            try:
                rem_raw.set_unit("eeg", "uV")
            except Exception:
                pass
        else:
            print(f"[{base}] MNE sans set_unit(): données bien en µV (métadonnée inchangée).")
    except Exception as e:
        print(f"[{base}] Échec scaling µV: {e} -> skip")
        return

    # I/O
    out_dir = out_root / base
    out_dir.mkdir(parents=True, exist_ok=True)
    if save_rem_fif:
        try:
            out_fif = out_dir / f"{base}_REM_concat_uV.fif"
            rem_raw.save(out_fif, overwrite=True)
            print(f"[{base}] Sauvé: {out_fif}")
        except Exception as e:
            print(f"[{base}] Échec save FIF: {e}")

    # TFR (Morlet) — epochs fixes
    try:
        epochs = mne.make_fixed_length_epochs(
            rem_raw, duration=float(epoch_dur), overlap=0.0, preload=True, verbose="ERROR"
        )
        # Ne garder que l'EEG dans epochs
        epochs.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False, misc=False)
    except Exception as e:
        print(f"[{base}] Échec création/filtrage epochs: {e} -> skip TFR")
        return

    freqs = np.linspace(float(freq_min), float(freq_max), int(n_freqs))
    n_cycles = freqs * float(cycles_mult)

    ch_names = epochs.ch_names
    print(f"[{base}] Canaux EEG pour TFR ({len(ch_names)}): {ch_names}")

    for ch in ch_names:
        out_png = out_dir / f"{base}_{_sanitize(ch)}_tfr_REM.png"
        if out_png.exists() and not overwrite_figs:
            print(f"[{base}:{ch}] [skip] {out_png.name} existe déjà.")
            continue

        try:
            power = mne.time_frequency.tfr_morlet(
                epochs, freqs=freqs, n_cycles=n_cycles,
                use_fft=True, return_itc=False, average=True,
                picks=[ch], decim=int(decim), verbose="ERROR"
            )
        except Exception as e:
            print(f"[{base}:{ch}] TFR erreur: {e} -> skip")
            continue

        # Autoscale robuste en dB
        try:
            Z = 10.0 * np.log10(np.maximum(power.data[0], np.finfo(float).tiny))
            if vmin is None or vmax is None:
                lo, hi = np.percentile(Z, list(auto_pct))
                vmin_eff, vmax_eff = float(lo), float(hi)
            else:
                vmin_eff, vmax_eff = float(vmin), float(vmax)
        except Exception as e:
            print(f"[{base}:{ch}] Autoscale erreur: {e} -> skip")
            del power
            continue

        # Plot + garde-fou "figure blanche"
        try:
            fig = power.plot(
                picks=[ch], dB=True, cmap=str(cmap),
                vmin=vmin_eff, vmax=vmax_eff,
                baseline=None, show=False
            )
        except TypeError:
            fig = power.plot(picks=[ch], dB=True, cmap=str(cmap),
                             baseline=None, show=False)
            figs_tmp = fig if isinstance(fig, (list, tuple)) else [fig]
            for f in figs_tmp:
                for ax in f.axes:
                    artists = list(ax.images) + [c for c in ax.collections if hasattr(c, "set_clim")]
                    for art in artists:
                        art.set_clim(vmin_eff, vmax_eff)

        figs = fig if isinstance(fig, (list, tuple)) else [fig]
        ax0 = figs[0].axes[0] if figs and figs[0].axes else None
        is_blank = (ax0 is None) or (len(ax0.images) == 0 and len(ax0.collections) == 0)

        if is_blank:
            print(f"[{base}:{ch}] figure vide -> skip (rien sauvegardé)")
            try:
                for f in figs:
                    plt.close(f)
            except Exception:
                pass
            del power
            continue

        if isinstance(fig, (list, tuple)):
            fig = fig[0]
        try:
            fig.savefig(out_png, dpi=200, bbox_inches="tight")
            print(f"[{base}:{ch}] [ok] {out_png.name}")
        except Exception as e:
            print(f"[{base}:{ch}] Save figure erreur: {e}")
        finally:
            plt.close(fig)
            del power

    print(f"[{base}] Terminé.")


# ===================== LANCEMENT BATCH =====================
if PATIENTS is None:
    PATIENTS = discover_patients(fif_root)
    bases_preview = [b for b, _ in PATIENTS]
    print(f"Patients détectés ({len(PATIENTS)}): {bases_preview}")

for item in PATIENTS:
    process_one_patient(item)


Patients détectés (78): ['AE129', 'AN166', 'BA152', 'BA171', 'BB114', 'BF181', 'BJ138', 'BJM190', 'BO60', 'CA169', 'CB165', 'CC175', 'CD164', 'CD28', 'CJP53', 'CM161', 'CP155', 'CS131', 'CS147', 'DA110', 'DA174', 'DBJ184', 'DI136', 'DJ137', 'DSJ112', 'EF130', 'FP144', 'GD170', 'GH163', 'GR108', 'GS191', 'GX111', 'HG167', 'IS179', 'JB173', 'JLJ177', 'JP141', 'KH113', 'LF126', 'LJ192', 'LM183', 'LS162', 'MA27', 'MC154', 'MFJ160', 'MG16', 'MHTK39', 'ML135', 'MM109', 'MN143', 'MP150', 'MP187', 'MRM132', 'MS128', 'NCJM193', 'NGA157', 'PA139', 'PB186', 'PF133', 'PJC140', 'PJL153', 'RB103', 'RD158', 'RG156', 'RH146', 'RJP148', 'SB176', 'SB178', 'SD134', 'SJP172', 'TG189', 'TJ127', 'TLM168', 'TM142', 'TM151', 'TO145', 'VA182', 'VJ149']

=== AE129 ===
<Raw | AE129_preprocessed_bip.fif, 33 x 9428992 (36832.0 s), ~2.32 GiB, data loaded>
[AE129] Segments REM valides: 391
<Raw | AE129_preprocessed_bip.fif, 33 x 3002880 (11730.0 s), ~756.1 MiB, data loaded>
[AE129] Montage bipolaire créé. Canaux con

/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:204: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)


[AE129] Canaux EEG retenus (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
[AE129] MNE sans set_unit(): données bien en µV (métadonnée inchangée).
Writing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/AE129/AE129_REM_concat_uV.fif


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:344: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/AE129/AE129_REM_concat_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


Closing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/AE129/AE129_REM_concat_uV.fif
[done]
[AE129] Sauvé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/AE129/AE129_REM_concat_uV.fif
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[AE129] Canaux EEG pour TFR (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[AE129:Fp2-C4] [ok] AE129_Fp2-C4_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[AE129:C4-O2] [ok] AE129_C4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[AE129:T4-O2] [ok] AE129_T4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied

...

No baseline correction applied
[TO145:Fp2-C4] [ok] TO145_Fp2-C4_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[TO145:C4-O2] [ok] TO145_C4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[TO145:T4-O2] [ok] TO145_T4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[TO145:Cz-Pz] [ok] TO145_Cz-Pz_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[TO145:Fp1-C3] [ok] TO145_Fp1-C3_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[TO145:C3-O1] [ok] TO145_C3-O1_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:204: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:344: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/VA182/VA182_REM_concat_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


No baseline correction applied
[VA182:Fp2-C4] [ok] VA182_Fp2-C4_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VA182:C4-O2] [ok] VA182_C4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VA182:T4-O2] [ok] VA182_T4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VA182:Cz-Pz] [ok] VA182_Cz-Pz_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VA182:Fp1-C3] [ok] VA182_Fp1-C3_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VA182:C3-O1] [ok] VA182_C3-O1_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:204: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/715951341.py:344: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/VJ149/VJ149_REM_concat_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


No baseline correction applied
[VJ149:Fp2-C4] [ok] VJ149_Fp2-C4_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VJ149:C4-O2] [ok] VJ149_C4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VJ149:T4-O2] [ok] VJ149_T4-O2_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VJ149:Cz-Pz] [ok] VJ149_Cz-Pz_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VJ149:Fp1-C3] [ok] VJ149_Fp1-C3_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
No baseline correction applied
[VJ149:C3-O1] [ok] VJ149_C3-O1_tfr_REM.png
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").


In [4]:
# === Debug FIF structure (à exécuter dans une nouvelle cellule) ===
from pathlib import Path
import re
import mne

# ⬇️ Modifie ce chemin si besoin (ex: un autre patient)
fif_path = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/full/AE129/AE129_preprocessed_bip.fif")

try:
    raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")
except Exception as e:
    print(f"Erreur lecture FIF: {e}")
    raise

print(raw)
print("sfreq:", raw.info.get("sfreq"), "| nchan:", raw.info.get("nchan"), "| meas_date:", raw.info.get("meas_date"))
print("bads:", raw.info.get("bads"))
try:
    print("montage:", raw.get_montage())
except Exception as e:
    print("montage: (non défini ou erreur)", e)

# Tableau (index, nom, type, unité) des 50 premiers canaux
types = raw.get_channel_types(unique=False)
units = [ch["unit"] for ch in raw.info["chs"]]

# Conversion code unité -> nom lisible
try:
    from mne.io.constants import FIFF
    _unit_map = {getattr(FIFF, k): k for k in dir(FIFF) if k.startswith("FIFF_UNIT_")}
except Exception:
    _unit_map = {}

def unit_name(u):
    return _unit_map.get(u, "NA" if u in (0, None) else str(u))

print("\n--- Channels (first 50) [idx, name, type, unit] ---")
for i, (nm, tp, u) in enumerate(zip(raw.ch_names, types, units)):
    if i >= 50: break
    print(f"{i:>3}  {nm:<20}  {tp:<6}  {unit_name(u)}")

# Ce que voit pick_types() à partir des métadonnées actuelles
picks_eeg = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
print("\nEEG picks by metadata:", picks_eeg, [raw.ch_names[p] for p in picks_eeg] if len(picks_eeg) else [])

# Heuristique: détecter les noms EEG standards (même si le type n'est pas 'eeg')
# On tolère espaces/underscore et préfixe 'EEG '
_std = r"^(A1|A2|M1|M2|Fp1|Fp2|Fz|F1|F2|F3|F4|F5|F6|F7|F8|FCz|FC[1-6]?|Cz|C1|C2|C3|C4|C5|C6|T3|T4|T5|T6|T7|T8|CPz|CP[1-6]?|Pz|P1|P2|P3|P4|P5|P6|P7|P8|POz|PO[1-8]?|Oz|Iz|O1|O2)$"
def norm_name(s: str) -> str:
    return s.replace("EEG ", "").replace("-REF", "").replace(" ", "").replace("_", "")
cand_eeg = [ch for ch in raw.ch_names if re.match(_std, norm_name(ch))]
print("\nEEG candidates by NAME (regex):", cand_eeg)

# Aperçu des annotations embarquées (si présentes)
ann = raw.annotations
if ann is not None and len(ann) > 0:
    from collections import Counter
    cnt = Counter([d.upper() for d in ann.description])
    print("\nAnnotations summary:", cnt)
else:
    print("\nAnnotations: aucune embarquée dans ce FIF")


<Raw | AE129_preprocessed_bip.fif, 33 x 9428992 (36832.0 s), ~32 KiB, data not loaded>
sfreq: 256.0 | nchan: 33 | meas_date: 2023-11-21 21:54:59+00:00
bads: []
montage: None

--- Channels (first 50) [idx, name, type, unit] ---
  0  Fp1                   eeg     FIFF_UNIT_V
  1  Fp2                   eeg     FIFF_UNIT_V
  2  C3                    eeg     FIFF_UNIT_V
  3  C4                    eeg     FIFF_UNIT_V
  4  T3                    eeg     FIFF_UNIT_V
  5  T4                    eeg     FIFF_UNIT_V
  6  O1                    eeg     FIFF_UNIT_V
  7  O2                    eeg     FIFF_UNIT_V
  8  EOGD                  eeg     FIFF_UNIT_V
  9  EOGG                  eeg     FIFF_UNIT_V
 10  A1                    eeg     FIFF_UNIT_V
 11  Menton                eeg     FIFF_UNIT_V
 12  VTH                   eeg     FIFF_UNIT_V
 13  VAB                   eeg     FIFF_UNIT_V
 14  THERM                 eeg     FIFF_UNIT_V
 15  NAF2P                 eeg     FIFF_UNIT_V
 16  RONF            

In [ ]:
# %% ------------------------------------------------------------
# PSD REM par patient + figure "une sous-figure par bande"
#        (source = .fif, montage bipolaire appliqué et sauvegardé)
# -------------------------------------------------------------
import os
from pathlib import Path
import numpy as np
import mne
import matplotlib.pyplot as plt

# ===================== PARAMÈTRES =====================
fif_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/full")  # {base}/*.fif ou *.fif
annot_root  = Path("/Volumes/Crucial X6/EEG/raw")                          # {base}/*.txt hypnogramme
out_root    = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")

# Laisser à None pour auto-découverte
# Peut aussi être une liste de bases ["AE129", ...] ou de tuples [("AE129","/chemin/fichier.fif"), ...]
PATIENTS    = None

# Bandes EEG
BANDS = {
    "Delta": (0.5, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}

# Paramètres Welch
psd_fmin   = 0.5
psd_fmax   = 45.0
psd_win_s  = 4.0   # durée de segment (s)
psd_ovlp   = 0.5   # chevauchement relatif (0..1)

# Seuils REM
MIN_REM_SEG_S   = psd_win_s   # on refuse les segments < fenêtre Welch
MIN_REM_TOTAL_S = 2 * psd_win_s  # durée totale minimale après filtrage

# Sauvegardes
overwrite_figs = False
save_rem_fif   = True   # sauvegarder le concat REM en .fif (bipolaire + µV)
# ===================== /PARAMS =====================


# ---- Import de la fonction d'annotations (fallback si besoin) ----
def _ensure_get_rem_annotations():
    try:
        from src.annotations import get_rem_annotations  # version projet
        return get_rem_annotations
    except Exception:
        import pandas as pd
        from pathlib import Path
        def load_annotation_file(txt_path):
            df = pd.read_csv(txt_path, sep="\t", names=["start", "temps", "stage", "index"])
            df = df.dropna(subset=["start", "stage"])
            df["duration"] = df["start"].shift(-1) - df["start"]
            df = df[:-1]
            rem_df = df[df["stage"].str.upper().str.strip() == "REM"]
            return rem_df[["start", "duration"]].values

        def get_rem_annotations(base_name, annot_dir):
            patient_code = base_name.split("_")[0]
            txt_dir = Path(annot_dir) / patient_code
            txt_candidates = list(txt_dir.glob("*.txt"))
            for txt_path in txt_candidates:
                try:
                    rem_intervals = load_annotation_file(txt_path)
                    if len(rem_intervals) > 0:
                        return mne.Annotations(
                            onset=[float(s) for s, _ in rem_intervals],
                            duration=[float(d) for _, d in rem_intervals],
                            description=["REM"] * len(rem_intervals)
                        )
                except Exception:
                    continue
            return None
        return get_rem_annotations

get_rem_annotations = _ensure_get_rem_annotations()


# ===================== I/O: découverte des FIF =====================
def discover_patients(fif_root: Path):
    """Retourne [(base, fif_path), ...] en cherchant des .fif."""
    mapping = {}

    # 1) sous-dossiers {base}/*.fif (prend le .fif le plus gros)
    for child in sorted(fif_root.iterdir()):
        if not child.is_dir():
            continue
        base = child.name
        fif_candidates = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if fif_candidates:
            mapping[base] = max(fif_candidates, key=lambda p: p.stat().st_size)

    # 2) fichiers *.fif directement à la racine
    root_fifs = [p for p in fif_root.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
    for p in root_fifs:
        base = p.stem.split("_")[0]
        cur = mapping.get(base)
        if cur is None or p.stat().st_size > cur.stat().st_size:
            mapping[base] = p

    return sorted(mapping.items())  # [(base, path), ...]


def _find_fif_for_base(base: str) -> Path | None:
    """Cherche un .fif pour `base` dans fif_root/{base}/*.fif, sinon à la racine."""
    child = fif_root / base
    if child.is_dir():
        cand = [p for p in child.glob("*.fif") if p.is_file() and not p.name.startswith("._")]
        if cand:
            return max(cand, key=lambda p: p.stat().st_size)
    cand = [p for p in fif_root.glob(f"{base}*.fif") if p.is_file() and not p.name.startswith("._")]
    if cand:
        return max(cand, key=lambda p: p.stat().st_size)
    return None


# ===================== BIPOLAIRE =====================
def _safe_bipolar(inst: mne.io.BaseRaw, anode: str, cathode: str, new_name: str, base: str) -> bool:
    """Crée un canal bipolaire anode-cathode ssi les deux existent (inst modifié en place)."""
    if anode in inst.ch_names and cathode in inst.ch_names:
        try:
            mne.set_bipolar_reference(
                inst, anode=anode, cathode=cathode, ch_name=new_name,
                drop_refs=False, copy=False, verbose="ERROR"
            )
            return True
        except Exception as e:
            print(f"[{base}] Bipolaire {new_name} échec: {e}")
    else:
        missing = [x for x in (anode, cathode) if x not in inst.ch_names]
        print(f"[{base}] Bipolaire {new_name} ignoré (manque {missing})")
    return False


def _apply_bipolar_montage(inst: mne.io.BaseRaw, base: str) -> None:
    """Applique MYMONTAGE_BIP + typage des canaux + réduction au set voulu."""
    # Paires à créer
    pairs = [
        ("EOGD", "A1", "EOGD-A1"),
        ("EOGG", "A1", "EOGG-A1"),
        ("Fp2",  "C4", "Fp2-C4"),
        ("C4",   "O2", "C4-O2"),
        ("T4",   "O2", "T4-O2"),
        ("Cz",   "Pz", "Cz-Pz"),
        ("Fp1",  "C3", "Fp1-C3"),
        ("C3",   "O1", "C3-O1"),
        ("Fp1",  "T3", "Fp1-T3"),
        ("T3",   "O1", "T3-O1"),
    ]

    created = []
    for a, c, n in pairs:
        if _safe_bipolar(inst, a, c, n, base):
            created.append(n)

    # Typage
    eeg_bip = ["Fp2-C4", "C4-O2", "T4-O2", "Cz-Pz", "Fp1-C3", "C3-O1", "Fp1-T3", "T3-O1"]
    eog_bip = ["EOGD-A1", "EOGG-A1"]
    keep_raw = ["Menton", "JAMBG", "JAMBD", "RONF", "EMG1", "EMG2", "ECG"]

    type_map = {}
    for ch in eeg_bip:
        if ch in inst.ch_names: type_map[ch] = "eeg"
    for ch in eog_bip:
        if ch in inst.ch_names: type_map[ch] = "eog"
    for ch in ("Menton", "EMG1", "EMG2", "JAMBG", "JAMBD"):
        if ch in inst.ch_names: type_map[ch] = "emg"
    if "ECG"  in inst.ch_names: type_map["ECG"]  = "ecg"
    if "RONF" in inst.ch_names: type_map["RONF"] = "misc"

    if type_map:
        try:
            inst.set_channel_types(type_map)
        except Exception as e:
            print(f"[{base}] set_channel_types après bipolaire: {e}")

    # Réduction stricte aux canaux désirés
    desired = eog_bip + eeg_bip + keep_raw
    present = [ch for ch in desired if ch in inst.ch_names]
    if present:
        inst.pick(present)
        print(f"[{base}] Montage bipolaire appliqué. Canaux ({len(inst.ch_names)}): {inst.ch_names}")
    else:
        print(f"[{base}] Aucun canal bipolaire/utile présent après montage.")


# ===================== PSD =====================
def compute_psd_db(raw_eeg: mne.io.BaseRaw, fmin: float, fmax: float,
                   win_s: float, ovlp: float):
    """Welch PSD en dB re µV²/Hz (raw_eeg doit déjà être en µV et limité aux EEG)."""
    sfreq = float(raw_eeg.info["sfreq"])
    n_times = int(raw_eeg.n_times)

    # fenêtre souhaitée en échantillons, bornée par la longueur du signal
    n_win = int(round(win_s * sfreq))
    n_per_seg = int(np.clip(n_win, 2, n_times))  # au moins 2 échantillons

    # overlap borné pour garantir noverlap < nperseg
    ovlp = float(np.clip(ovlp, 0.0, 0.99))
    n_overlap = int(np.floor(ovlp * n_per_seg))
    n_overlap = min(max(n_overlap, 0), n_per_seg - 1)

    try:
        spec = raw_eeg.compute_psd(
            method="welch", fmin=fmin, fmax=fmax,
            n_per_seg=n_per_seg, n_overlap=n_overlap,
            picks=None, reject_by_annotation=False, verbose="ERROR",
        )
    except ValueError:
        print("[WARN] Welch a échoué avec overlap; on réessaie sans recouvrement.")
        spec = raw_eeg.compute_psd(
            method="welch", fmin=fmin, fmax=fmax,
            n_per_seg=min(n_per_seg, int(raw_eeg.n_times)), n_overlap=0,
            picks=None, reject_by_annotation=False, verbose="ERROR",
        )

    freqs = spec.freqs
    psd   = spec.get_data()  # µV²/Hz
    psd_db = 10.0 * np.log10(np.maximum(psd, np.finfo(float).tiny))
    return freqs, psd_db  # shape: (n_ch, n_freqs)


def plot_psd_by_bands(freqs, psd_db, ch_names, bands, out_png, title_prefix=""):
    """Figure avec une sous-figure par bande: moyenne + p10/p90."""
    f_lo, f_hi = float(freqs[0]), float(freqs[-1])
    bands_eff = {k: (max(v[0], f_lo), min(v[1], f_hi))
                 for k, v in bands.items() if min(v[1], f_hi) > max(v[0], f_lo)}

    n_b = len(bands_eff)
    if n_b == 0:
        print("Aucune bande dans la passband -> pas de figure.")
        return False

    n_cols = 2
    n_rows = int(np.ceil(n_b / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 3.2 * n_rows), sharex=False)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n_b:]:
        ax.axis("off")

    for idx, (band_name, (bmin, bmax)) in enumerate(bands_eff.items()):
        ax = axes[idx]
        mask = (freqs >= bmin) & (freqs <= bmax)
        if not np.any(mask):
            ax.set_visible(False)
            continue

        F = freqs[mask]
        M = psd_db[:, mask]              # (n_ch, n_freqs_band)
        mean_curve = M.mean(axis=0)
        p10 = np.percentile(M, 10, axis=0)
        p90 = np.percentile(M, 90, axis=0)

        ax.semilogx(F, mean_curve, lw=1.5)
        ax.fill_between(F, p10, p90, alpha=0.25, linewidth=0)

        ax.set_title(f"{band_name}  [{bmin:.1f}–{bmax:.1f}] Hz")
        ax.set_xlabel("Fréquence (Hz)")
        ax.set_ylabel("PSD (dB re µV²/Hz)")
        ax.grid(True, which="both", alpha=0.3)

    fig.suptitle(f"{title_prefix}PSD REM — moy ± p10–p90 par bande", y=0.995)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    return True


# ===================== PIPELINE =====================
def process_one_patient(item):
    # item: "base" (str) ou (base, fif_path)
    if isinstance(item, tuple):
        base, fif_path = item
        fif_path = Path(fif_path)
    else:
        base = str(item)
        fif_path = _find_fif_for_base(base)

    if fif_path is None or not fif_path.exists():
        print(f"[{base}] FIF introuvable -> skip")
        return

    # === Early skip si sorties présentes ===
    out_dir = out_root / base
    out_fif = out_dir / f"{base}_REM_concat_bip_uV.fif"
    out_png = out_dir / f"{base}_PSD_bands_REM_bip.png"
    if out_png.exists() and (not save_rem_fif or out_fif.exists()) and not overwrite_figs:
        msg = f"[{base}] Sorties déjà présentes -> skip ({out_png.name}"
        if save_rem_fif:
            msg += f", {out_fif.name}"
        msg += ")"
        print(msg)
        return

    print(f"\n=== {base} ===")
    try:
        # ⚠️ Nécessaire pour le montage bipolaire
        raw_full = mne.io.read_raw_fif(fif_path, preload=True, verbose="ERROR")
    except Exception as e:
        print(f"[{base}] Erreur lecture FIF: {e} -> skip")
        return
    print(raw_full)

    # Annotations REM via .txt
    rem_annots = get_rem_annotations(base, annot_dir=str(annot_root))
    if rem_annots is None or len(rem_annots) == 0:
        print(f"[{base}] Aucune annotation REM -> skip")
        return

    sfreq = float(raw_full.info["sfreq"])
    t_end = raw_full.times[-1]
    eps   = 1.0 / sfreq

    # Segments REM valides (bornés au signal)
    valid = []
    for onset, dur, desc in zip(rem_annots.onset, rem_annots.duration, rem_annots.description):
        if str(desc).upper() != "REM" or dur is None or dur <= 0:
            continue
        tmin = max(0.0, float(onset))
        tmax = min(tmin + float(dur), t_end) - eps
        if tmax > tmin:
            valid.append((tmin, tmax))

    if not valid:
        print(f"[{base}] Aucun segment REM valide -> skip")
        return

    # PRINT des durées + filtrage des segments trop courts
    durations = [tmax - tmin for (tmin, tmax) in valid]
    total_dur = float(np.sum(durations))
    print(f"[{base}] Segments REM valides: {len(valid)} | total={total_dur:.2f}s | min={np.min(durations):.2f}s | median={np.median(durations):.2f}s | max={np.max(durations):.2f}s")

    kept = []
    for (tmin, tmax) in valid:
        dur = tmax - tmin
        if dur < MIN_REM_SEG_S:
            print(f"[{base}]  - skip segment {tmin:.2f}-{tmax:.2f}s (durée {dur:.2f}s < {MIN_REM_SEG_S:.2f}s)")
        else:
            kept.append((tmin, tmax))

    if not kept or sum(tmax - tmin for (tmin, tmax) in kept) < MIN_REM_TOTAL_S:
        print(f"[{base}] Durée REM après filtrage insuffisante (< {MIN_REM_TOTAL_S:.2f}s) -> skip")
        return

    # Extraire & concaténer
    rem_raws = []
    for (tmin, tmax) in kept:
        try:
            seg = raw_full.copy().crop(tmin=tmin, tmax=tmax, verbose="ERROR")
            rem_raws.append(seg)
        except Exception as e:
            print(f"[{base}] Crop {tmin:.2f}-{tmax:.2f} échoué: {e}")

    if not rem_raws:
        print(f"[{base}] Rien à concaténer -> skip")
        return

    rem_raw = mne.concatenate_raws(rem_raws, verbose="ERROR")

    # ===== Montage bipolaire puis typage =====
    _apply_bipolar_montage(rem_raw, base)

    # === Ne garder que les EEG bipolaires ===
    try:
        rem_raw.pick_types(meg=False, eeg=True, eog=False, ecg=False, emg=False, stim=False,
                           misc=False, resp=False, seeg=False, ecog=False, fnirs=False)
        if len(rem_raw.ch_names) == 0:
            print(f"[{base}] Aucun canal EEG bipolaire -> skip")
            return
        print(f"[{base}] Canaux EEG bipolaires ({len(rem_raw.ch_names)}): {rem_raw.ch_names}")
    except Exception as e:
        print(f"[{base}] Échec du filtrage EEG-only: {e} -> skip")
        return

    # === Mise en µV (toujours) ===
    try:
        rem_raw.load_data()
        eeg_picks = mne.pick_types(rem_raw.info, eeg=True, meg=False, eog=False, ecg=False, emg=False)
        rem_raw.apply_function(lambda x: x * 1e6, picks=eeg_picks, channel_wise=True)  # V -> µV
        if hasattr(rem_raw, "set_unit"):
            try:
                rem_raw.set_unit("eeg", "uV")
            except Exception:
                pass
    except Exception as e:
        print(f"[{base}] Échec scaling µV: {e} -> skip")
        return

    # === Sauvegarde REM concat + bipolaire ===
    out_dir.mkdir(parents=True, exist_ok=True)
    if save_rem_fif:
        try:
            rem_raw.save(out_fif, overwrite=True)
            print(f"[{base}] Sauvegardé: {out_fif}")
        except Exception as e:
            print(f"[{base}] Save FIF échoué: {e}")

    # === Early skip figure si déjà présente ===
    if out_png.exists() and not overwrite_figs:
        print(f"[{base}] Figure existe déjà -> skip: {out_png.name}")
        return

    # === PSD (Welch) en dB ===
    freqs, psd_db = compute_psd_db(rem_raw, psd_fmin, psd_fmax, psd_win_s, psd_ovlp)

    # === Figure "une sous-figure par bande" ===
    ok = plot_psd_by_bands(freqs, psd_db, rem_raw.ch_names, BANDS, out_png, title_prefix=f"{base} — ")
    if ok:
        print(f"[{base}] Figure OK: {out_png.name}")
    else:
        print(f"[{base}] Aucune bande plot -> rien sauvegardé")

    print(f"[{base}] Terminé.")


# ===================== LANCEMENT =====================
if PATIENTS is None:
    PATIENTS = discover_patients(fif_root)
    print(f"Patients détectés ({len(PATIENTS)}): {[b for b, _ in PATIENTS]}")

for item in PATIENTS:
    process_one_patient(item)


Patients détectés (78): ['AE129', 'AN166', 'BA152', 'BA171', 'BB114', 'BF181', 'BJ138', 'BJM190', 'BO60', 'CA169', 'CB165', 'CC175', 'CD164', 'CD28', 'CJP53', 'CM161', 'CP155', 'CS131', 'CS147', 'DA110', 'DA174', 'DBJ184', 'DI136', 'DJ137', 'DSJ112', 'EF130', 'FP144', 'GD170', 'GH163', 'GR108', 'GS191', 'GX111', 'HG167', 'IS179', 'JB173', 'JLJ177', 'JP141', 'KH113', 'LF126', 'LJ192', 'LM183', 'LS162', 'MA27', 'MC154', 'MFJ160', 'MG16', 'MHTK39', 'ML135', 'MM109', 'MN143', 'MP150', 'MP187', 'MRM132', 'MS128', 'NCJM193', 'NGA157', 'PA139', 'PB186', 'PF133', 'PJC140', 'PJL153', 'RB103', 'RD158', 'RG156', 'RH146', 'RJP148', 'SB176', 'SB178', 'SD134', 'SJP172', 'TG189', 'TJ127', 'TLM168', 'TM142', 'TM151', 'TO145', 'VA182', 'VJ149']
[AE129] Sorties déjà présentes -> skip (AE129_PSD_bands_REM_bip.png, AE129_REM_concat_bip_uV.fif)
[AN166] Sorties déjà présentes -> skip (AN166_PSD_bands_REM_bip.png, AN166_REM_concat_bip_uV.fif)
[BA152] Sorties déjà présentes -> skip (BA152_PSD_bands_REM_bip.pn

/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:176: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:397: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BA171/BA171_REM_concat_bip_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


[BA171] Sauvegardé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BA171/BA171_REM_concat_bip_uV.fif
[BA171] Figure OK: BA171_PSD_bands_REM_bip.png
[BA171] Terminé.

=== BB114 ===
<Raw | BB114_preprocessed_bip.fif, 33 x 7678720 (29995.0 s), ~1.89 GiB, data loaded>
[BB114] Segments REM valides: 184 | total=5519.28s | min=30.00s | median=30.00s | max=30.00s
[BB114] Montage bipolaire appliqué. Canaux (17): ['EOGD-A1', 'EOGG-A1', 'Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1', 'Menton', 'JAMBG', 'JAMBD', 'RONF', 'EMG1', 'EMG2', 'ECG']
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[BB114] Canaux EEG bipolaires (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
Writing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BB114/BB114_REM_concat_bip_uV.fif
Closing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BB114/BB114_REM_concat_bip_uV.fif


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:176: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:397: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BB114/BB114_REM_concat_bip_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


[done]
[BB114] Sauvegardé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BB114/BB114_REM_concat_bip_uV.fif
[BB114] Figure OK: BB114_PSD_bands_REM_bip.png
[BB114] Terminé.

=== BF181 ===
<Raw | BF181_preprocessed_bip.fif, 33 x 7099648 (27733.0 s), ~1.75 GiB, data loaded>
[BF181] Segments REM valides: 173 | total=5189.32s | min=30.00s | median=30.00s | max=30.00s
[BF181] Montage bipolaire appliqué. Canaux (17): ['EOGD-A1', 'EOGG-A1', 'Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1', 'Menton', 'JAMBG', 'JAMBD', 'RONF', 'EMG1', 'EMG2', 'ECG']
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[BF181] Canaux EEG bipolaires (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
Writing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BF181/BF181_REM_concat_bip_uV.fif
Closing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BF181/BF181_REM_concat_bip_uV.fif


...

[done]
[BF181] Sauvegardé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BF181/BF181_REM_concat_bip_uV.fif
[BF181] Figure OK: BF181_PSD_bands_REM_bip.png
[BF181] Terminé.

=== BJ138 ===
<Raw | BJ138_preprocessed_bip.fif, 33 x 7496192 (29282.0 s), ~1.84 GiB, data loaded>
[BJ138] Segments REM valides: 62 | total=1859.76s | min=30.00s | median=30.00s | max=30.00s
[BJ138] Montage bipolaire appliqué. Canaux (17): ['EOGD-A1', 'EOGG-A1', 'Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1', 'Menton', 'JAMBG', 'JAMBD', 'RONF', 'EMG1', 'EMG2', 'ECG']
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[BJ138] Canaux EEG bipolaires (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
Writing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BJ138/BJ138_REM_concat_bip_uV.fif
Closing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BJ138/BJ138_REM_concat_bip_uV.fif
[done]


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:176: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:397: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BJ138/BJ138_REM_concat_bip_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


[BJ138] Sauvegardé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BJ138/BJ138_REM_concat_bip_uV.fif
[BJ138] Figure OK: BJ138_PSD_bands_REM_bip.png
[BJ138] Terminé.

=== BJM190 ===
<Raw | BJM190_preprocessed_bip.fif, 11 x 13112320 (25610.0 s), ~1.07 GiB, data loaded>
[BJM190] Aucune annotation REM -> skip

=== BO60 ===
<Raw | BO60_preprocessed_bip.fif, 33 x 719872 (2812.0 s), ~181.3 MiB, data loaded>
[BO60] Segments REM valides: 11 | total=2862.95s | min=7.00s | median=30.00s | max=1705.99s
[BO60] Montage bipolaire appliqué. Canaux (17): ['EOGD-A1', 'EOGG-A1', 'Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1', 'Menton', 'JAMBG', 'JAMBD', 'RONF', 'EMG1', 'EMG2', 'ECG']
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[BO60] Canaux EEG bipolaires (8): ['Fp2-C4', 'C4-O2', 'T4-O2', 'Cz-Pz', 'Fp1-C3', 'C3-O1', 'Fp1-T3', 'T3-O1']
Writing /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BO60/BO60_REM_concat_bip_uV.fif
Closing /Volumes/

/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:176: RuntimeWarning: The unit for channel(s) RONF has changed from V to NA.
  inst.set_channel_types(type_map)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_13039/1891302959.py:397: RuntimeWarning: This filename (/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BO60/BO60_REM_concat_bip_uV.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  rem_raw.save(out_fif, overwrite=True)


[BO60] Sauvegardé: /Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD/BO60/BO60_REM_concat_bip_uV.fif
[BO60] Figure OK: BO60_PSD_bands_REM_bip.png
[BO60] Terminé.

=== CA169 ===
<Raw | CA169_preprocessed_bip.fif, 38 x 10131456 (39576.0 s), ~2.87 GiB, data loaded>
[CA169] Segments REM valides: 198 | total=5939.23s | min=30.00s | median=30.00s | max=30.00s


## Par canal

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
REM-only FIF -> (1) PSD par canal (sous-figures) + (2) Topomaps de puissance par bande.

- Auto-découvre les patients dans rem_root: rem_only/{base}/{base}_REM_concat*.fif
- Charge le FIF, garde uniquement les EEG, suppose déjà en µV (configurable).
- Welch PSD par canal, figure multi-sous-graphes (une PSD / canal, bandes ombrées).
- Intègre la PSD par bande -> topomaps (auto-montage standard_1020 si possible).
- Sauvegarde 2 PNG par patient dans rem_only/{base}/.

À coller dans Jupyter ou exécuter comme script.
"""

from pathlib import Path
import re
import numpy as np
import mne
import matplotlib.pyplot as plt

# ===================== PARAMÈTRES =====================
rem_root = Path("/Volumes/Crucial X6/EEG/preprocessed/bipolaire/PWD")

# Si None -> auto-détection de tous les patients ayant *_REM_concat*.fif
PATIENTS = None   # ex: ["AE129","BJ138"]

# Le FIF est-il déjà en µV ? (True si tu as utilisé ton pipeline précédent)
assume_already_uV = True

# Bornes PSD
psd_fmin, psd_fmax = 1.0, 45.0

# Welch
psd_win_s = 4.0          # longueur de fenêtre en secondes
psd_ovlp  = 0.5          # chevauchement (0..1)

# Bandes d'intérêt
BANDS = {
    "Delta": (0.5, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}

# Aesthetics
max_cols = 5             # max colonnes pour la grille "PSD par canal"
shade_alpha = 0.08       # opacité des bandes ombrées sur PSD
psd_fig_dpi = 180
topo_fig_dpi = 180
topo_cmap = "viridis"    # colormap pour topomaps
# ===================== /PARAMS =====================


def discover_patients_from_fif(root: Path):
    """Cherche rem_only/{base}/{base}_REM_concat*.fif et retourne la liste des 'base'."""
    bases = []
    for p in sorted(root.glob("*/*_REM_concat*.fif")):
        base = p.parent.name
        if base not in bases:
            bases.append(base)
    return bases


def find_patient_fif(root: Path, base: str) -> Path | None:
    """Retourne le premier .fif correspondant au patient."""
    cands = sorted((root / base).glob(f"{base}_REM_concat*.fif"))
    return cands[0] if cands else None


def normalize_eeg_names(raw: mne.io.BaseRaw):
    """Renomme canaux pour mieux coller au 10-20: 'EEG Fp1'->'Fp1', 'T3'->'T7', etc."""
    mapping = {}
    for ch in raw.ch_names:
        new = ch
        # remove leading "EEG " ou "EEG_"
        new = re.sub(r"^EEG[\s_]+", "", new)
        # espaces -> rien
        new = new.replace(" ", "")
        # anciennes dénominations
        repl = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
        if new in repl:
            new = repl[new]
        # parfois 'Fpz'/'FpZ' etc. -> standardiser la casse: première lettre maj + reste tel quel
        # (on laisse MNE faire le matching le plus souple possible)
        mapping[ch] = new
    raw.rename_channels(mapping)


def set_montage_if_possible(raw: mne.io.BaseRaw):
    """Tente d'appliquer un montage standard_1020 pour permettre les topomaps."""
    try:
        normalize_eeg_names(raw)
        montage = mne.channels.make_standard_montage("standard_1020")
        # sur EDF, il peut rester des canaux non EEG -> on ne garde que l'EEG
        raw.pick("eeg")
        raw.set_montage(montage, on_missing="ignore")
        # check: combien ont des positions
        pos_cnt = sum([ch["loc"] is not None and np.any(ch["loc"][:3]) for ch in raw.info["chs"]])
        if pos_cnt < len(raw.ch_names) // 2:
            print(f"  [montage] Avertissement: peu de positions reconnues ({pos_cnt}/{len(raw.ch_names)})")
        else:
            print(f"  [montage] Positions connues pour {pos_cnt}/{len(raw.ch_names)} canaux.")
    except Exception as e:
        print(f"  [montage] Impossible d'appliquer standard_1020: {e}")


def compute_welch_psd(raw: mne.io.BaseRaw, fmin, fmax, win_s, ovlp):
    sfreq = float(raw.info["sfreq"])
    n_per_seg = max(2, min(int(sfreq * win_s), raw.n_times))
    n_overlap = int(n_per_seg * float(ovlp))
    psd = raw.compute_psd(
        method="welch", fmin=float(fmin), fmax=float(fmax),
        n_per_seg=n_per_seg, n_overlap=n_overlap,
        picks="eeg", verbose="ERROR"
    )
    freqs = psd.freqs
    data = psd.get_data()         # (n_channels, n_freqs), µV^2/Hz si raw en µV
    return freqs, data


def plot_psd_by_channel(base: str, freqs, psd_data, out_dir: Path):
    """Une sous-figure par canal, PSD en dB, bandes ombrées."""
    n_ch, n_f = psd_data.shape
    n_cols = min(max_cols, n_ch)
    n_rows = int(np.ceil(n_ch / n_cols))

    fig_w = 3.2 * n_cols
    fig_h = 2.4 * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)

    # pré-calcul dB
    tiny = np.finfo(float).tiny
    psd_db = 10.0 * np.log10(np.maximum(psd_data, tiny))

    for i in range(n_ch):
        r, c = divmod(i, n_cols)
        ax = axes[r, c]
        ax.semilogx(freqs, psd_db[i], lw=1.0)
        ax.set_xlim(freqs[0], freqs[-1])
        if r == n_rows - 1:
            ax.set_xlabel("Fréquence (Hz)")
        if c == 0:
            ax.set_ylabel("PSD (dB re µV²/Hz)")
        ax.set_title(raw_eeg_names[i], fontsize=9)
        ax.grid(True, alpha=0.25)

        # bandes ombrées
        for (f1, f2) in BANDS.values():
            f1p, f2p = max(f1, freqs[0]), min(f2, freqs[-1])
            if f2p > f1p:
                ax.axvspan(f1p, f2p, color="k", alpha=shade_alpha)

    # cache axes vides
    for j in range(n_ch, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        axes[r, c].axis("off")

    plt.suptitle(f"{base} — PSD par canal (REM)", y=1.02, fontsize=14)
    plt.tight_layout()
    out_png = out_dir / f"{base}_PSD_by_channel_REM.png"
    fig.savefig(out_png, dpi=psd_fig_dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [save] {out_png.name}")


def band_power_from_psd(freqs, psd_data, band):
    """Intègre la PSD sur [fmin, fmax] -> puissance par canal (µV²)."""
    f1, f2 = band
    idx = np.where((freqs >= f1) & (freqs < f2))[0]
    if idx.size == 0:
        return np.zeros(psd_data.shape[0])
    return np.trapz(psd_data[:, idx], freqs[idx], axis=1)


def plot_topomaps(base: str, raw_info, freqs, psd_data, out_dir: Path):
    """Topomap de la puissance par bande (en dB)."""
    # calcule puissance µV² par bande
    band_vals = {}
    for name, (f1, f2) in BANDS.items():
        P = band_power_from_psd(freqs, psd_data, (f1, f2))           # µV²
        P_db = 10.0 * np.log10(np.maximum(P, np.finfo(float).tiny))  # dB re µV²
        band_vals[name] = P_db

    # figure (N bandes)
    n_b = len(BANDS)
    fig, axes = plt.subplots(1, n_b, figsize=(3.2 * n_b, 3.0), squeeze=False)
    axes = axes[0]

    for ax, (name, vals) in zip(axes, band_vals.items()):
        im, cn = mne.viz.plot_topomap(
            vals, raw_info, axes=ax, show=False, cmap=topo_cmap, contours=0, sphere="auto"
        )
        ax.set_title(name)
        mne.viz.utils._add_colorbar(cn, im, ax)

    plt.suptitle(f"{base} — Topomaps de puissance (dB) par bande (REM)", y=1.04, fontsize=14)
    plt.tight_layout()
    out_png = out_dir / f"{base}_Topomaps_band_power_REM.png"
    fig.savefig(out_png, dpi=topo_fig_dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"  [save] {out_png.name}")


# ===================== BOUCLE PATIENTS =====================
if PATIENTS is None:
    PATIENTS = discover_patients_from_fif(rem_root)
print(f"Patients détectés ({len(PATIENTS)}): {PATIENTS}")

for base in PATIENTS:
    fif_path = find_patient_fif(rem_root, base)
    if fif_path is None:
        print(f"[{base}] pas de *_REM_concat*.fif -> skip")
        continue

    print(f"\n=== {base} ===")
    out_dir = rem_root / base
    out_dir.mkdir(parents=True, exist_ok=True)

    # Charge le FIF
    try:
        raw = mne.io.read_raw_fif(fif_path, preload=False, verbose="ERROR")
    except Exception as e:
        print(f"  [read] erreur: {e} -> skip")
        continue

    # Garde uniquement les EEG
    raw.pick("eeg")
    raw.load_data()

    # rescale vers µV si nécessaire
    if not assume_already_uV:
        raw.apply_function(lambda x: x * 1e6, picks="eeg", channel_wise=True)

    # Appliquer une montage si possible (pour topomaps)
    set_montage_if_possible(raw)

    # noms des canaux EEG (après normalisation)
    raw_eeg_names = raw.ch_names

    # PSD (Welch)
    freqs, psd = compute_welch_psd(raw, psd_fmin, psd_fmax, psd_win_s, psd_ovlp)

    # Figure 1: PSD par canal (sous-figures)
    plot_psd_by_channel(base, freqs, psd, out_dir)

    # Figure 2: Topomaps de puissance par bande
    try:
        plot_topomaps(base, raw.info, freqs, psd, out_dir)
    except Exception as e:
        print(f"  [topomap] impossible de tracer: {e} (positions manquantes ?)")


Patients détectés (67): ['AE129', 'AN166', 'BA152', 'BA171', 'BB114', 'BF181', 'BJ138', 'BO60', 'CA169', 'CB165', 'CC175', 'CD164', 'CD28', 'CJP53', 'CM161', 'CP155', 'CS131', 'CS147', 'DA110', 'DA174', 'DI136', 'DJ137', 'DSJ112', 'EF130', 'FP144', 'GD170', 'GH163', 'GR108', 'GS191', 'HG167', 'IS179', 'JB173', 'JLJ177', 'JP141', 'KH113', 'LS162', 'MA27', 'MC154', 'MFJ160', 'MG16', 'MHTK39', 'ML135', 'MM109', 'MN143', 'MP150', 'MRM132', 'MS128', 'NGA157', 'PA139', 'PF133', 'PJC140', 'RB103', 'RD158', 'RG156', 'RH146', 'RJP148', 'SB176', 'SB178', 'SD134', 'SJP172', 'TJ127', 'TLM168', 'TM142', 'TM151', 'TO145', 'VA182', 'VJ149']

=== AE129 ===
Reading 0 ... 3002879  =      0.000 ... 11729.996 secs...
  [montage] Positions connues pour 11/11 canaux.
  [save] AE129_PSD_by_channel_REM.png
  [topomap] impossible de tracer: module 'mne.viz.utils' has no attribute '_add_colorbar' (positions manquantes ?)

=== AN166 ===
Reading 0 ... 675839  =      0.000 ...  2639.996 secs...
  [montage] Positio

/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:173: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd_data[:, idx], freqs[idx], axis=1)
/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:191: RuntimeWarning: Only 10 head digitization points of the specified kinds ("eeg", "extra",), fitting may be inaccurate
  im, cn = mne.viz.plot_topomap(


  [save] AN166_PSD_by_channel_REM.png
  [topomap] impossible de tracer: module 'mne.viz.utils' has no attribute '_add_colorbar' (positions manquantes ?)

=== BA152 ===
Reading 0 ... 890879  =      0.000 ...  3479.996 secs...
  [montage] Positions connues pour 11/11 canaux.


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:191: RuntimeWarning: Only 10 head digitization points of the specified kinds ("eeg", "extra",), fitting may be inaccurate
  im, cn = mne.viz.plot_topomap(


  [save] BA152_PSD_by_channel_REM.png
  [topomap] impossible de tracer: module 'mne.viz.utils' has no attribute '_add_colorbar' (positions manquantes ?)

=== BA171 ===
Reading 0 ... 460799  =      0.000 ...  1799.996 secs...
  [montage] Positions connues pour 11/11 canaux.


...

/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:191: RuntimeWarning: Only 10 head digitization points of the specified kinds ("eeg", "extra",), fitting may be inaccurate
  im, cn = mne.viz.plot_topomap(


  [save] VA182_PSD_by_channel_REM.png
  [topomap] impossible de tracer: module 'mne.viz.utils' has no attribute '_add_colorbar' (positions manquantes ?)

=== VJ149 ===
Reading 0 ... 399359  =      0.000 ...  1559.996 secs...
  [montage] Positions connues pour 11/11 canaux.


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:191: RuntimeWarning: Only 10 head digitization points of the specified kinds ("eeg", "extra",), fitting may be inaccurate
  im, cn = mne.viz.plot_topomap(


  [save] VJ149_PSD_by_channel_REM.png
  [topomap] impossible de tracer: module 'mne.viz.utils' has no attribute '_add_colorbar' (positions manquantes ?)


/var/folders/mm/2jz7s_g55bv92cw9dt44jn6c0000gn/T/ipykernel_10977/789711730.py:191: RuntimeWarning: Only 10 head digitization points of the specified kinds ("eeg", "extra",), fitting may be inaccurate
  im, cn = mne.viz.plot_topomap(
